# DEMO 9: EXPLAIN, Query Profile and Query Insights

In Power BI, you use **DAX Studio** to see how queries execute (Storage Engine scans vs Formula Engine calculations). In Databricks, we use `EXPLAIN` and the **Query Profile** to understand exactly how Spark executes your SQL.

In this demo, you’ll learn:
- How to read a query plan with `EXPLAIN`
- What key operators mean (Scan, Filter, Aggregate, Join, Shuffle)
- How to use the **Query Profile** in the SQL Editor after running a query
- How to spot **Photon** acceleration in your plans

> **Note:** Run the setup cell below (Cell 2) to create the demo tables, then open the **SQL Editor** (click "SQL Editor" in the left sidebar) to run the remaining queries there. The SQL Editor gives you the full Query Profile and Query Insights experience.

---

In [0]:
%run ./demo_9_setup

## Part 1: EXPLAIN — See the Query Plan Without Running the Query

Put `EXPLAIN` before any SQL statement to see **how** Spark would execute it — without actually running it. This is like DAX Studio’s query plan view.

The plan reads **bottom-up**:
- **Bottom** = where data enters (file scans)
- **Top** = where results exit

| Operator | What it does | What to look for |
| --- | --- | --- |
| **PhotonScan** / **FileScan** | Reads data from Delta Lake files | Check `DataFilters` — filters should appear here (pushdown) |
| **PhotonGroupingAgg** | GROUP BY aggregation | Normal for reporting queries |
| **PhotonBroadcastHashJoin** | Small table broadcast join | Good — small table sent to all workers |
| **ShuffleExchange** | Data redistributed between workers | Expensive — watch for large shuffles |
| **PhotonTopK** / **Sort** | ORDER BY / LIMIT | Can be expensive on large datasets |

> Copy each SQL query below into a **new tab** in the SQL Editor and run it there.

---

In [0]:
%sql
-- ============================================================
-- EXPLAIN: Simple aggregation with filter
-- ============================================================
-- Look at the PhotonScan at the bottom: notice the DataFilters
-- include our WHERE condition. This means the filter was "pushed
-- down" to the scan level — Spark reads less data.

EXPLAIN
SELECT region, ROUND(SUM(amount), 2) AS total_revenue
FROM sales_fact
WHERE region = 'North'
GROUP BY region;

## Reading the Plan: Filter Pushdown

In the output above, look for `DataFilters:` inside the `PhotonScan` line. You should see:

```
DataFilters: [isnotnull(region), (region = North)]
```

This means the filter was **pushed down** to the scan — Spark skips files/rows that don’t match before any aggregation happens. This is the Delta Lake equivalent of VertiPaq’s segment elimination.

Also note: `ReadSchema: struct<region:string,amount:double>` — only the 2 columns we need are read. The other 6 columns are **pruned** (not read from storage). This is called **column pruning**.

---

In [0]:
%sql
-- ============================================================
-- EXPLAIN: Full table scan (no filter)
-- ============================================================
-- Compare this plan to the one above. Without a WHERE clause,
-- there are NO DataFilters — Spark must read ALL files.
-- This is the most common performance mistake in dashboard queries.

EXPLAIN
SELECT region, ROUND(SUM(amount), 2) AS total_revenue
FROM sales_fact
GROUP BY region;

## Part 2: JOIN Plans — Broadcast vs Shuffle

When you join two tables, Spark chooses a strategy based on table sizes:

| Join Type | When Used | Performance |
| --- | --- | --- |
| **BroadcastHashJoin** | One table is small (≤ 10MB) | Fast — small table is copied to all workers |
| **SortMergeJoin** | Both tables are large | Slower — both tables must be shuffled and sorted |

In Power BI, this is like the difference between a relationship to a small lookup table (fast) vs joining two large fact tables (expensive).

Our `dim_products` table (20 rows) will trigger a **BroadcastHashJoin** — look for it in the plan below.

---

In [0]:
%sql
-- ============================================================
-- EXPLAIN: JOIN — look for BroadcastHashJoin
-- ============================================================
-- The small dim_products table gets broadcast to all workers.
-- Look for "PhotonBroadcastHashJoin" in the plan.
-- Also notice: the date filter is pushed down to the sales_fact scan.

EXPLAIN
SELECT 
  s.region, 
  p.product_name, 
  SUM(s.amount) AS revenue
FROM sales_fact s
INNER JOIN dim_products p ON s.product_category = p.product_category
WHERE s.sale_date >= '2024-01-01'
GROUP BY s.region, p.product_name
ORDER BY revenue DESC
LIMIT 10;

## Part 3: The Query Profile (Visual Execution Plan)

While `EXPLAIN` shows the plan **before** execution, the **Query Profile** shows what **actually happened** after a query runs.

**How to access the Query Profile in the SQL Editor:**
1. Run a query in the SQL Editor
2. After execution, click the **elapsed time link** at the bottom of the results pane (e.g. "2.3s, 20 rows")
3. Click **"See query profile"** to open the full DAG
4. You’ll see a visual DAG (Directed Acyclic Graph) showing:
   - Each operator as a node (click any node for detailed metrics)
   - Toggle between **Time spent**, **Memory peak**, and **Rows** metrics at the top
   - Color intensity shows which operators took the longest
   - Purple nodes = **Photon** (fast); grey nodes = Spark fallback

**What to look for:**
- Which operator took the **longest**? (darkest color)
- Are operators processing **way more rows** than expected?
- Is the query **fully Photon**? (all purple = best performance)
- Click **"See longest operators"** for a quick summary of bottlenecks

---

In [0]:
%sql
-- ============================================================
-- Run this in the SQL Editor, then check the Query Profile
-- ============================================================
-- After running, click the elapsed time link at the bottom of
-- the results pane (e.g. "2.3s, 20 rows"), then click
-- "See query profile" to open the visual DAG.

SELECT 
  s.region,
  s.product_category,
  COUNT(*) AS order_count,
  ROUND(SUM(s.amount), 2) AS total_revenue,
  ROUND(AVG(s.amount), 2) AS avg_order_value
FROM sales_fact s
INNER JOIN dim_products p ON s.product_category = p.product_category
WHERE s.sale_date >= '2024-06-01'
  AND s.channel = 'online'
GROUP BY s.region, s.product_category
ORDER BY total_revenue DESC;

## Part 4: Photon — The Fast Lane

At the bottom of every `EXPLAIN` output, you’ll see a **Photon Explanation** section:

```
== Photon Explanation ==
The query is fully supported by Photon.
```

**Photon** is Databricks’ native C++ vectorized engine that processes data in batches of thousands of rows at once (instead of row-by-row). It provides up to 5x better performance for:
- Large aggregations
- Join-heavy queries
- Write operations (INSERT, UPDATE, MERGE)

**Power BI parallel:** Think of Photon as the equivalent of VertiPaq’s in-memory columnar scan speed, but without the memory ceiling.

If you see `"The query is NOT fully supported by Photon"`, it means some operators fell back to the standard Spark engine (shown in grey in the Query Profile).

---

In [0]:
%sql
-- ============================================================
-- EXPLAIN FORMATTED: more readable output
-- ============================================================
-- EXPLAIN FORMATTED gives a more structured view of the plan.
-- Look at the "Photon Explanation" section at the bottom.

EXPLAIN FORMATTED
SELECT 
  sale_date,
  region,
  SUM(amount) AS daily_revenue,
  SUM(SUM(amount)) OVER (
    PARTITION BY region 
    ORDER BY sale_date
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS cumulative_revenue
FROM sales_fact
WHERE sale_date >= '2024-01-01'
GROUP BY sale_date, region
ORDER BY region, sale_date;

## Part 5: Query History and Insights

When running queries on a **SQL Warehouse** (which is what your dashboards use), Databricks provides **Query Insights** — an automatic performance analysis:

- **Query History** — see all queries that ran, their duration, and who ran them
- **Query Profile** — the visual DAG for any past query
- **Bottleneck Detection** — automatic highlighting of slow operators
- **Recommendations** — suggestions like “add a filter” or “reduce columns”

**How to access Query History:**
1. Click **"Query History"** in the left sidebar (under the SQL section)
2. You'll see a list of all executed queries with status, duration, and user
3. Click any query text to open the **query details panel** on the right

**What you'll find in the details panel:**
- **Query status** — Queued, Running, Finished, Failed, or Cancelled
- **Query metrics** — rows returned, data scanned, pruning % (filter icons show how much data was skipped)
- **Wall-clock duration** — breakdown of scheduling, optimization/pruning, and execution time
- **DAG preview** — click **"See query profile"** for the full visualization
- **Longest operators** — click to jump straight to the bottleneck
- **I/O details** — data read and written during execution

**Pro tip:** Use Query History to compare the same query before and after adding filters — you'll see the difference in data scanned and duration immediately.

---

---

## Summary: What to Look For

| Signal | Good | Bad |
| --- | --- | --- |
| **DataFilters in Scan** | Filters pushed down (reads less data) | Empty DataFilters = full table scan |
| **ReadSchema** | Only needed columns listed | All columns = wasted I/O |
| **BroadcastHashJoin** | Small table broadcast (fast) | SortMergeJoin on a table that should be small |
| **Photon operators** | All operators are Photon (purple) | Grey operators = Spark fallback (slower) |
| **Shuffle/Exchange** | Minimal shuffles | Large shuffles = network bottleneck |
| **Rows processed** | Decreasing at each stage (filters working) | Same row count throughout = no filtering happening |

**The key takeaway:** Always include filters (WHERE clauses) in your dashboard queries, select only the columns you need, and use `EXPLAIN` to verify your filters are being pushed down to the scan level. This is the single biggest performance win for BI practitioners.